In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q13/q13_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# 1) Count orders per customer (exclude special requests)
cust_order_counts = (
    orders
    .loc[
        ~orders["O_COMMENT"].str.contains(r"special[\S|\s]*requests"),
        "O_CUSTKEY"
    ]
    .value_counts()
)

# 2) Map counts back to every customer, fill missing with 0 and cast to int
order_counts = (
    customer["C_CUSTKEY"]
    .map(cust_order_counts)
    .fillna(0)
    .astype(int)
)

# 3) Get distribution of customers by their order count
dist = order_counts.value_counts()

# 4) Build the final DataFrame and sort by CUSTDIST desc, then C_COUNT desc
total = (
    dist
    .rename("CUSTDIST")        # name the counts column
    .reset_index()
)
# rename the columns in one go to avoid a KeyError
total.columns = ["C_COUNT", "CUSTDIST"]

# sort by customer distribution and then by count
total = total.sort_values(["CUSTDIST", "C_COUNT"], ascending=[False, False])
